# Week 14 — Parallel and Randomized Algorithms

This is the worked reference notebook: run live in lecture, fully solved.
The version students receive with TODOs in place of the solved parts is
`assignments/pds/a14-parallel-randomized/starter/parallel_randomized.py`.

## 1. Amdahl's Law and a real parallel map (Lecture 1)

In [1]:
def amdahl_speedup(p, n_workers):
    return 1 / ((1 - p) + p / n_workers)

def amdahl_ceiling(p):
    return 1 / (1 - p)

# 30% sequential -> p = 0.7 parallelizable
p = 0.7
assert round(amdahl_speedup(p, 1), 4) == 1.0
assert round(amdahl_ceiling(p), 4) == round(1/0.3, 4)
print('Ceiling speedup at p=0.7:', round(amdahl_ceiling(p), 3))
print('Speedup at 4 workers:', round(amdahl_speedup(p, 4), 3))
print('Speedup at 100 workers:', round(amdahl_speedup(p, 100), 3))

Ceiling speedup at p=0.7: 3.333
Speedup at 4 workers: 2.105
Speedup at 100 workers: 3.257


## 2. Monte Carlo primality testing: Miller-Rabin (Lecture 2)

In [2]:
import random

def witness_proves_composite(n, a, d, r):
    x = pow(a, d, n)
    if x == 1 or x == n - 1:
        return False
    for _ in range(r - 1):
        x = pow(x, 2, n)
        if x == n - 1:
            return False
    return True

def is_probably_prime(n, k=20):
    if n < 2:
        return False
    for p in [2, 3, 5, 7, 11, 13]:
        if n % p == 0:
            return n == p
    d, r = n - 1, 0
    while d % 2 == 0:
        d //= 2
        r += 1
    for _ in range(k):
        a = random.randrange(2, n - 1)
        if witness_proves_composite(n, a, d, r):
            return False
    return True

# Exact trace from Lecture 2: n=221, d=55, r=2
assert witness_proves_composite(221, 174, 55, 2) == False   # does NOT prove composite
assert witness_proves_composite(221, 137, 55, 2) == True    # PROVES composite

# Second trace: n=561 (Carmichael number), d=35, r=4, witness a=2
assert witness_proves_composite(561, 2, 35, 4) == True

for p in [17, 97, 7919]:
    assert is_probably_prime(p)
for c in [221, 1000, 8051, 561]:
    assert not is_probably_prime(c)
print('Miller-Rabin checks passed, matching Lecture 2\'s exact traces')

Miller-Rabin checks passed, matching Lecture 2's exact traces


## 3. Parallel sorting and the reduction pattern (Lecture 3)

In [3]:
def merge(a, b):
    result, i, j = [], 0, 0
    while i < len(a) and j < len(b):
        if a[i] <= b[j]:
            result.append(a[i]); i += 1
        else:
            result.append(b[j]); j += 1
    result.extend(a[i:]); result.extend(b[j:])
    return result

def merge_sort(arr):
    if len(arr) <= 1:
        return arr
    mid = len(arr) // 2
    return merge(merge_sort(arr[:mid]), merge_sort(arr[mid:]))

arr = [38, 27, 43, 3, 9, 82, 10]
assert merge_sort(arr) == sorted(arr)

def partial_sum(chunk):
    return sum(chunk)

data = list(range(1, 100_001))
chunks = [data[i::4] for i in range(4)]
partials = [partial_sum(c) for c in chunks]
parallel_total = sum(partials)
assert parallel_total == sum(data) == 5_000_050_000

def chunk_sum_and_count(chunk):
    return sum(chunk), len(chunk)

def parallel_average(data, chunks):
    partials = [chunk_sum_and_count(c) for c in chunks]
    total = sum(s for s, _ in partials)
    count = sum(c for _, c in partials)
    return total / count

avg_data = [10, 20, 30, 5, 15]
avg_chunks = [[10, 20, 30], [5, 15]]
assert parallel_average(avg_data, avg_chunks) == sum(avg_data) / len(avg_data) == 16.0

def partial_max(chunk):
    return max(chunk)

max_data = [random.randint(0, 10**9) for _ in range(100_000)]
max_chunks = [max_data[i::4] for i in range(4)]
assert max(partial_max(c) for c in max_chunks) == max(max_data)
print('Merge sort and reduction-pattern checks passed, matching Lecture 3 exactly')

Merge sort and reduction-pattern checks passed, matching Lecture 3 exactly


## 4. Randomized selection: quickselect (Lecture 4)

In [4]:
def quickselect(arr, k):
    if len(arr) == 1:
        return arr[0]
    pivot = random.choice(arr)
    lows = [x for x in arr if x < pivot]
    highs = [x for x in arr if x > pivot]
    pivots = [x for x in arr if x == pivot]
    if k < len(lows):
        return quickselect(lows, k)
    elif k < len(lows) + len(pivots):
        return pivot
    else:
        return quickselect(highs, k - len(lows) - len(pivots))

def quickselect_median(arr):
    return quickselect(arr, len(arr) // 2)

qs_arr = [7, 10, 4, 3, 20, 15]
qs_sorted = sorted(qs_arr)
for k in range(len(qs_arr)):
    assert quickselect(qs_arr, k) == qs_sorted[k]

median_arr = [9, 1, 8, 2, 7, 3, 6]
assert quickselect(median_arr, len(median_arr) // 2) == sorted(median_arr)[len(median_arr) // 2] == 6

for test in [[5, 3, 1, 4, 2], [9, 9, 1], [100], [7, 2, 9, 4, 6], [3, 3, 3, 3, 3]]:
    assert quickselect_median(test) == sorted(test)[len(test) // 2]
print('Quickselect checks passed, matching Lecture 4\'s exact traces')

Quickselect checks passed, matching Lecture 4's exact traces
